In [27]:
import torch
import time
from tokenizer import load_rules_tokens, Tokenizer
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.utils.data import random_split

In [28]:
PADDING_TOKEN = 0
SOS_TOKEN = 1
EOS_TOKEN = 2
UNK_TOKEN = 3

In [29]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device of type: {device.type}')

Using device of type: cuda


In [30]:
def load_data(filename, num_samples=10000):
    lines = open(filename, encoding='utf-8').read().strip().split('\n')

    en_texts = []
    fr_texts = []
    if isinstance(num_samples, float):
        assert 0. < num_samples <= 1., 'For float num_samples, it has to be in (0, 1]'
        num_samples = int(len(lines)*num_samples)
    
    assert isinstance(num_samples, int), f'num_samples is of wrong type: {type(num_samples)}'
    assert num_samples >= 0, 'num_samples can\'t be negative'

    for line in lines[:num_samples]:
        parts = line.split('\t')
        if len(parts) < 2: continue
        
        en_texts.append(parts[0])
        fr_texts.append(parts[1])
    
    return en_texts, fr_texts

In [50]:
def collate_fn(batch):
    inputs, targets = zip(*batch)
    inputs_padded = nn.utils.rnn.pad_sequence(inputs, batch_first=True, padding_value=PADDING_TOKEN)
    targets_padded = nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=PADDING_TOKEN)
    
    return inputs_padded, targets_padded

In [32]:
class Vocab:
    def __init__(self):
        self.token2index = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UKN>": 3}
        self.index2char = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UKN>"}
        self.n_chars = 4
    
    def add_tokens(self, tokens):
        for token in tokens:
            if token not in self.token2index:
                self.token2index[token] = self.n_chars
                self.index2char[self.n_chars] = token
                self.n_chars += 1

In [42]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1, embedding_dim=32, drop_out=0.2):
        super().__init__()
        self.embedder = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers=num_layers, 
                            dropout=drop_out, batch_first=True)
    
    def forward(self, X, hidden=None):
        embs = self.embedder(X)
        out, hidden = self.lstm(embs, hidden)
        return out, hidden

In [44]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1, embedding_dim=32, drop_out=0.2):
        super().__init__()
        self.embedder = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.dropout = nn.Dropout(drop_out)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, 
                            batch_first=True, dropout=drop_out)
        self.out = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, X, hidden):
        embs = self.embedder(X)
        output, hidden = self.lstm(embs, hidden)
        output = self.dropout(output)
        preds = self.out(output)
        
        return preds, hidden

In [48]:
class TranslateDataset(Dataset):
    def __init__(self, X, Y, in_vocab, out_vocab):
        self.X = X
        self.Y = Y
        self.in_vocab = in_vocab
        self.out_vocab = out_vocab
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        X, y = self.X[idx], self.Y[idx]
        input_indices = [self.in_vocab.token2index[t] for t in X] + [EOS_TOKEN]
        target_indices = [SOS_TOKEN] + [self.out_vocab.token2index[t] for t in y] + [EOS_TOKEN]
        return torch.tensor(input_indices), torch.tensor(target_indices)

In [60]:
class Seq2Seq:
    def __init__(self, in_vocab, out_vocab, hidden_size, num_layers=1, embbeding_dim=32, drop_out=0.2):
        self.in_vocab = in_vocab
        self.out_vocab = out_vocab
        
        self.encoder = Encoder(in_vocab.n_chars, hidden_size, num_layers, embbeding_dim, drop_out).to(device)
        self.decoder = Decoder(out_vocab.n_chars, hidden_size, num_layers, embbeding_dim, drop_out).to(device)
        
    def train(self, X, y, epochs=32, patience=5, min_delta=0.01, lr=0.01, batch_size=64):
        dataset = TranslateDataset(X, y, self.in_vocab, self.out_vocab)
        
        val_size = int(0.1 * len(dataset))
        train_size = len(dataset) - val_size
        train_ds, val_ds = random_split(dataset, [train_size, val_size])
        
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
        
        criterion = nn.CrossEntropyLoss(ignore_index=PADDING_TOKEN) 
        enc_op = torch.optim.AdamW(self.encoder.parameters(), lr=lr)
        dec_op = torch.optim.AdamW(self.decoder.parameters(), lr=lr)
        
        best_val_loss = float('inf')
        epochs_no_impr = 0
        
        start_time = time.time()
        print('Starting training...')
        for epoch in range(1, epochs+1):
            self.encoder.train(); self.decoder.train()
            train_loss = 0
            for inputs, targets in train_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                enc_op.zero_grad(); dec_op.zero_grad()
                
                _, hidden = self.encoder(inputs)
                
                predictions, _ = self.decoder(targets[:, :-1], hidden)
                
                loss = criterion(predictions.reshape(-1, self.out_vocab.n_chars),
                                targets[:, 1:].reshape(-1))
                
                loss.backward()
                enc_op.step(); dec_op.step()
                train_loss += loss.item()
                
            self.encoder.eval(); self.decoder.eval()
            val_loss = 0
            with torch.no_grad():
                for inputs, targets in val_loader:
                    inputs, targets = inputs.to(device), targets.to(device)
                    _, hidden = self.encoder(inputs)
                    
                    cur_batch_size = targets.size(0)
                    target_length = targets.size(1)

                    all_predictions = []

                    decoder_input = torch.full((cur_batch_size, 1), SOS_TOKEN, device=device, dtype=torch.long)
                    for i in range(1, target_length):
                        predictions, hidden = self.decoder(decoder_input, hidden)
                        all_predictions.append(predictions)

                        decoder_input = predictions.argmax(dim=-1).detach()
                    predictions_combined = torch.cat(all_predictions, dim=1)

                    loss = criterion(predictions_combined.reshape(-1, self.out_vocab.n_chars), targets[:, 1:].reshape(-1))
                    val_loss += loss.item()
                    
            avg_val_loss = val_loss / len(val_loader)
            print(f"[{time.time()-start_time}] Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f}")

            # -- EARLY STOPPING --
            if avg_val_loss < (best_val_loss - min_delta):
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                # self.save("best_model.pth", verbose=False)
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f"Early stopping: нет значимого улучшения ({min_delta}) в течение {patience} эпох.")
                    # self.load("best_model.pth")
                    break


In [37]:
en_tockenizer = Tokenizer(*load_rules_tokens('./en-rules.pkl', './en-tokens.pkl'))
fr_tockenizer = Tokenizer(*load_rules_tokens('./fr-rules.pkl', './fr-tokens.pkl'))

In [38]:
en_texts, fr_texts = load_data('./fra.txt', 1.)

In [39]:
en_tokenized = en_tockenizer.tokenize_texts(en_texts)
fr_tokenized = fr_tockenizer.tokenize_texts(fr_texts)

In [40]:
in_vocab = Vocab()
in_vocab.add_tokens(en_tockenizer.valid_chars_)

out_vocab = Vocab()
out_vocab.add_tokens(fr_tockenizer.valid_chars_)

In [61]:
model = Seq2Seq(in_vocab, out_vocab, 128, 4)
model.train(en_tokenized, fr_tokenized, batch_size=128)

Starting training...
[47.708478689193726] Epoch 2/32 | Train Loss: 4.7513 | Val Loss: 7.2842
[95.49889636039734] Epoch 3/32 | Train Loss: 4.0703 | Val Loss: 7.1477
[143.48270678520203] Epoch 4/32 | Train Loss: 3.7793 | Val Loss: 7.5291
[191.55558323860168] Epoch 5/32 | Train Loss: 3.5992 | Val Loss: 7.7501
[239.50714993476868] Epoch 6/32 | Train Loss: 3.4870 | Val Loss: 7.8934
[287.63708567619324] Epoch 7/32 | Train Loss: 3.3895 | Val Loss: 7.7878
[335.85929012298584] Epoch 8/32 | Train Loss: 3.3045 | Val Loss: 7.9009
Early stopping: нет значимого улучшения (0.01) в течение 5 эпох.
